# MIGA — Algorithm Overview
**Paper:** Figueroa-García, Neruda & Hernandez-Pérez (2023).
*A genetic algorithm for multivariate missing data imputation.*
Information Sciences 619, 947–967.  DOI: 10.1016/j.ins.2022.11.037

This notebook walks through the mathematical foundation and pseudocode
of MIGA so that every design decision in the reimplementation can be
traced back to a specific definition or equation in the paper.

## 1. Problem Statement

Given an incomplete data matrix **X** (n × p) with some entries missing,
find imputed values that minimise the statistical differences between the
**available** sub-matrix **X_A** (complete rows) and the **completed**
sub-matrix **X_C** (rows with at least one missing value, now filled in).

Two key challenges not handled by classical methods (EM, regression):
- Mixed variable types: *continuous*, *discrete*, *binary*
- Multiple missing observations across multiple variables simultaneously

## 2. Similarity between matrices (Definition 2)

Two matrices **X**, **Y** are *similar* if:

$$\mathscr{D}_r(\bar{\mathbf{x}}, \bar{\mathbf{y}}) \to 0$$
$$\mathscr{D}_r(\mathbf{S}_X, \mathbf{S}_Y) \to 0$$
$$\mathscr{D}_r(\mathbf{b}_X, \mathbf{b}_Y) \to 0$$

where $\mathscr{D}_r$ is the Minkowski $r$-norm distance (Definition 1):

$$\mathscr{D}_r(\mathbf{A}, \mathbf{B}) = \left(\sum_i\sum_j |a_{ij} - b_{ij}|^r\right)^{1/r}$$

For $r = \infty$: $\mathscr{D}_\infty(\mathbf{A}, \mathbf{B}) = \max_{i,j}|a_{ij}-b_{ij}|$

## 3. Fitness Function (Definition 5, Eq. 10)

To make the three goal distances **dimensionless and additive**, the paper
standardises means and covariances:

| Symbol | Formula | Purpose |
|--------|---------|---------|
| $\tilde{x}_A = \bar{x}_A / S_p$ | standardised means of $X_A$ | removes units from means |
| $\tilde{x}_C = \bar{x}_C / S_p$ | standardised means of $X_C$ | same |
| $\tilde{S} = S_A^{-1/2} S_C S_A^{-1/2}$ | relative covariance | $\tilde{S}=I \iff S_A = S_C$ |
| $\hat{b} = b_A - b_C$ | skewness difference | zero iff skewness preserved |

Pooled variance (Eq. 7): $S_p^2 = \frac{\text{dg}(S_A)\nu_A + \text{dg}(S_C)\nu_C}{\nu_T}$

The single fitness function to minimise:

$$\mathscr{F}_r := \min \left( \mathscr{D}_r(\tilde{x}_A, \tilde{x}_C)
+ \mathscr{D}_r(\tilde{S},\, I)
+ \mathscr{D}_r(b_A, b_C) \right)$$

$\mathscr{F}_r = 0 \iff \bar{x}_A=\bar{x}_C,\; S_A=S_C,\; b_A=b_C$

## 4. Individual Encoding (Definition 3)

- **Individual** $p_i$: a flat vector of length $k = |\mathbf{M}|$
  where $\mathbf{M}$ is the set of all missing positions $(i,j)$.
- Genes are *j-ordered*: consecutive genes in $p_i$ for the same column $j$
  form a "subset" — enabling variable-specific crossover.
- Each gene is sampled from a **random variable generator** $R_j$
  appropriate for the type of variable $j$
  (Normal, Discrete Uniform, Poisson, Exponential, …).

## 5. Population Structure per Generation

| Component | Size | Description |
|-----------|------|-------------|
| Elite | $c$ | Best $c$ individuals carried forward (elitism) |
| Mutation | $c_1 \times c_3$ | $c_3$ children per parent, from best $c_1$ |
| Crossover | $2(c_2-1)$ | Swap variable $j'$ between consecutive pairs |
| Diversity | $l - c - c_1 c_3 - 2(c_2-1)$ | Fresh random individuals |
| **Total** | **$l$** | Population size |

**Constraint**: $l > c + c_1 c_3 + 2(c_2-1)$

## 6. Algorithm Pseudocode (Algorithm 1)

In [ ]:
# Algorithm 1 — reproduced as Python pseudocode for clarity

def MIGA_pseudocode(X, X_A, M, l, G, c, c1, c2, c3, Q, r):
    """
    X   : incomplete dataset (n x p), NaN = missing
    X_A : complete rows of X
    M   : list of (row, col) missing positions  [j-ordered]
    l   : population size
    G   : number of generations per run
    c   : elite size
    c1  : mutation parent pool size
    c2  : crossover parent pool size
    c3  : mutation children per parent
    Q   : number of independent runs
    r   : Minkowski order
    """
    # Pre-compute available statistics
    mean_A, cov_A, skew_A = compute_stats(X_A)       # x̄_A, S_A, b_A

    best_overall = None
    for q in range(Q):
        P = initialize_population(l, k=len(M))       # random initial pop

        for g in range(G):
            scores = [F_r(individual, X_A, X_C) for individual in P]

            # Selection (elitism)
            elite = top_c(P, scores, c)

            # Operators
            mutants  = mutate(P, scores, c1, c3)     # c1*c3 new
            children = crossover(P, scores, c2)       # 2*(c2-1) new
            fresh    = random_individuals(n_div)      # diversity

            P = elite + mutants + children + fresh    # next generation
            # update best individual

        best_per_run = best_in(P)

    return best_overall_across_runs

print("Pseudocode displayed. See miga/core.py for the full implementation.")

## 7. Fitness Function Demo

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import numpy as np
from miga.statistics import compute_stats, pooled_std, relative_cov, minkowski_distance

# Small 2-variable example (matches paper Example 2)
np.random.seed(0)
n_A, n_C = 80, 20
X_A = np.column_stack([np.random.normal(5, 1, n_A), np.random.normal(3, 2, n_A)])
X_C = np.column_stack([np.random.normal(5.1, 1.05, n_C), np.random.normal(3.1, 2.1, n_C)])

nu_A, nu_C = n_A - 1, n_C - 1
mean_A, cov_A, skew_A = compute_stats(X_A)
mean_C, cov_C, skew_C = compute_stats(X_C)
S_p = pooled_std(cov_A, cov_C, nu_A, nu_C)

x_tilde_A = mean_A / S_p
x_tilde_C = mean_C / S_p
S_tilde   = relative_cov(cov_A, cov_C)
I         = np.eye(2)

r = np.inf
d_means = minkowski_distance(x_tilde_A, x_tilde_C, r)
d_cov   = minkowski_distance(S_tilde, I, r)
d_skew  = minkowski_distance(skew_A, skew_C, r)
F_r     = d_means + d_cov + d_skew

print(f"D_∞(x̃_A, x̃_C) = {d_means:.4f}  (means)")
print(f"D_∞(S̃,  I)     = {d_cov:.4f}  (covariance)")
print(f"D_∞(b_A, b_C)  = {d_skew:.4f}  (skewness)")
print(f"F_∞            = {F_r:.4f}")